# Итоговый проект от Мастерской


- Автор: Игнатьев Вячеслав
- Дата: 20.09.2026


## Описание проекта

Вы работаете аналитиком данных в Яндекс Такси. Каждый день компания обрабатывает миллионы поездок, оплаченных разными способами. Чтобы финансовая и продуктовая команды регулярно получали актуальные данные о выручке и поведении пассажиров, коллеги решили настроить автоматическую обработку этих данных.

Ваша задача — построить витрину данных, которая будет агрегировать информацию о поездках по каждому способу оплаты. Для этого нужно написать PySpark-скрипт, который рассчитает ключевые показатели: количество поездок, среднюю стоимость поездки, средние чаевые и суммарную выручку по каждому типу оплаты.

Чтобы процесс был полностью автоматическим и не зависел от ручных запусков, необходимо создать DAG в Airflow. Этот DAG должен ежедневно:

* проверять наличие новых файлов с данными;

* запускать Spark-задачу;

* формировать обновлённую итоговую таблицу.

Эта таблица станет основой для финансовых отчётов и аналитических дашбордов.


## Описание данных

Таблица `taxi_data` содержит данные об активности пользователей и состоит из следующих полей:

* `taxi_id` — идентификатор водителя;

* `trip_start_timestamp` — время начала поездки;

* `trip_end_timestamp` — время окончания поездки;

* `trip_seconds` — длительность поездки в секундах;

* `trip_miles` — дистанция поездки;

* `fare` — стоимость поездки;

* `tips` — размер чаевых;

* `trip_total` — общая стоимость поездки: стоимость поездки + чаевые + комиссия;

* `payment_type` — способ оплаты.


## Что нужно сделать

В проекте вам предстоит автоматизировать подготовку витрины данных по поездкам Яндекс Такси:

1. Сначала необходимо написать Spark-скрипт, который будет обрабатывать данные о поездках и агрегировать показатели по способам оплаты `payment_type`. Понадобится рассчитать несколько показателей:

* количество поездок, которое показывает общий спрос и загрузку сервиса;
* среднюю стоимость `fare`, которое отражает уровень среднего чека поездки;
* средние чаевые `tips` — индикатор удовлетворённости клиентов и мотивации водителей;
* суммарную выручку `trip_total` — ключевой показатель дохода компании.

Все результаты должны собираться в одну итоговую таблицу `taxi_payment_summary`. После этого таблицу нужно записать в ClickHouse с помощью JDBC-драйвера.

2. Далее вам понадобится настроить DAG в Airflow. DAG должен запускаться ежедневно. Перед запуском он проверяет наличие файла с данными за нужную дату в S3-хранилище и только после появления файла запускает Spark-задачу.


## Шаг 1. Настройте Spark-агрегацию


Параметры подключения передаются через переменные окружения и Airflow Connections.


Данные хранятся в формате Parquet, поэтому для чтения используйте метод `spark.read.parquet()`. Это быстрее и надёжнее, чем CSV.

Сгруппируйте данные по полю `payment_type` и рассчитайте четыре показателя:

* количество поездок — `count(*)`;
* среднюю стоимость — `avg(...)`;
* средние чаевые — `avg(...)`;
* суммарную выручку — `sum(...)`.

Так получится витрина для анализа информации по каждому способу оплаты. После этого настройте запись полученной таблицы в ClickHouse. Проверьте, что всё работает, и переходите в шагу 2.


In [1]:
# filename=my_spark_job.py
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os

# Создаём Spark-сессию
spark = (
SparkSession.builder.appName("myAggregateTest")
.config("fs.s3a.endpoint", "storage.yandexcloud.net")
.config("spark.dynamicAllocation.enabled", "false")
.config("spark.executor.instances", "1")
.config("spark.executor.cores", "1")
.config("spark.executor.memory", "1g")
.config("spark.driver.memory", "1g")
.getOrCreate()
)

# Указываем порт и параметры кластера ClickHouse
jdbcPort = 8443
jdbcHostname = os.environ["CLICKHOUSE_HOST"]
username = os.environ["CLICKHOUSE_USER"]
jdbcDatabase = "playground_" + username
jdbcUrl = f"jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true"


# Считываем Parquet-файл
parquet_path = "s3a://da-plus-dags/project_04/taxi_data.parquet"
df_from_parquet = spark.read.parquet(parquet_path)

# Строим агрегат по типам платежей
result_df = df_from_parquet.groupBy("payment_type").agg(
    F.count("*").alias("trips_count"),
    F.avg("fare").alias("avg_fare"),
    F.avg("tips").alias("avg_tips"),
    F.sum("trip_total").alias("total_revenue"))

result_df.write.format("jdbc") \
    .option("url", jdbcUrl) \
    .option("user", username) \
    .option("password", os.environ["CLICKHOUSE_PASSWORD"]) \
    .option("dbtable", "taxi_payment_summary") \
    .mode('overwrite') \
    .save()


## Шаг 2. Настройте DAG

Дату в качестве параметра передавать не нужно. В DAG используйте `S3KeySensor`, чтобы дождаться появления файла в S3. После этого запускайте `DataprocCreatePysparkJobOperator`, передав путь к вашему скрипту. Дополнительный класс-оператор создавать не требуется.


Параметры подключения передаются через переменные окружения и Airflow Connections.


In [ ]:
# filename=taxi_payment_dag.py
from datetime import datetime
from airflow import DAG
from airflow.models import Variable
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator


DAG_ID = "taxi_payment"

with DAG(
    DAG_ID,
    schedule='@daily',
    start_date=datetime(2025, 1, 1),
    catchup=False
) as dag:
    wait_for_input = S3KeySensor(
        task_id = 'wait_for_input',
        bucket_name = "da-plus-dags",
        bucket_key="project_04/taxi_data.parquet",
        poke_interval=300,
        timeout=3600,
        aws_conn_id='s3',
        mode='poke',
        wildcard_match=False,
    # Заполните необходимые параметры
    )
    run_pyspark = DataprocCreatePysparkJobOperator(
        name="taxi_payment_analysis_job",
        task_id="run_pyspark",
        cluster_id=Variable.get("dataproc_cluster_id"),
        main_python_file_uri=Variable.get("taxi_spark_job_uri"),
    )
    wait_for_input >> run_pyspark


## Шаг 3. Запустите DAG с помощью Airflow UI

Теперь можно переходить к запуску. Нажмите кнопку «Проверить», подождите 5 минут и снова нажмите её. Вам будут показаны данные для входа в веб-интерфейс Airflow. В интерфейсе найдите ваш DAG и запустите его.

Проверьте, что DAG выполнился, а результат соответствует ожиданиям. Если всё получилось — поздравляем, проект завершён!


payment_type|trips_count|avg_fare  |avg_tips    |total_revenue|
------------+-----------+----------+------------+-------------+
Cash        |    1410155| 11.194248|0.0033251946|     16971150|
Credit Card |    1108843| 15.869293|    3.477157|     23160130|
Dispute     |       1842| 13.327953|         0.0|     27052.29|
No Charge   |      12843|13.5359535|  0.12569416|     184374.6|
Prcard      |        968| 11.172262|  0.25764462|     11513.15|
Unknown     |       5180| 11.791628|  0.35713595|     67725.07|
Way2ride    |          3|       3.5|  0.79333335|        14.38|
Pcard       |        878|  9.833428|  0.18314351|      9175.55|
